In [2]:
# ===== AwareLiquid-Tiny 一键训练(自包含,含最新修复)=====
import os, sys, subprocess, json, torch

assert torch.cuda.is_available(), "没GPU!代码执行程序→更改运行时类型→GPU,再重跑"
print("GPU:", torch.cuda.get_device_name(0))

# 1) 挂 Drive(弹授权就点同意;已挂过会直接过)
from google.colab import drive
drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/awareliquid_tiny'
DATA, CKPT = f'{ROOT}/data', f'{ROOT}/checkpoints'
METRICS=f'{ROOT}/metrics.jsonl'
os.makedirs(DATA, exist_ok=True); os.makedirs(CKPT, exist_ok=True)

# 2) 干净拉最新代码(带 multimodal.py + label 修复)
import shutil; shutil.rmtree('/content/M1', ignore_errors=True)
subprocess.run(['git','clone','--depth','1','https://github.com/everest-an/M1.git','/content/M1'], check=True)
os.chdir('/content/M1'); sys.path.insert(0,'/content/M1')
subprocess.run(['git','-C','/content/M1','log','--oneline','-1'])  # 应显示 e6b33c3

# 3) 装依赖(锁住 Colab 的 torch)
tp=torch.__version__.split('+')[0]
open('/content/c.txt','w').write(f'torch=={tp}\n')
subprocess.run([sys.executable,'-m','pip','install','-q','-c','/content/c.txt',
    'datasets','transformers','tokenizers','tqdm','einops'], check=True)

# 4) 分词(已在 Drive 则跳过)
if not os.path.exists(f'{DATA}/meta.json'):
    print("分词中(约10-20分钟,只第一次)...")
    tok=('import prepare_data;from types import SimpleNamespace;'
         'prepare_data.main(SimpleNamespace(dataset="roneneldan/TinyStories",'
         f'config=None,tokenizer="gpt2",out_dir={DATA!r}))')
    subprocess.run([sys.executable,'-c',tok], check=True, cwd='/content/M1')
print(json.load(open(f'{DATA}/meta.json')))

# 5) 断点续训
cands=[f'{CKPT}/{n}' for n in ('last.pt','final.pt')]
cands+=[f'{CKPT}/{f}' for f in os.listdir(CKPT) if f.startswith('ckpt_') and f.endswith('.pt')]
ex=[c for c in cands if os.path.exists(c)]
resume=['--resume', max(ex,key=os.path.getmtime)] if ex else []
print("续训:", resume or "全新开始")

# 6) 训练
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
cmd=[sys.executable,'train.py','--data_dir',DATA,'--ckpt_dir',CKPT,
 '--metrics_jsonl',METRICS,'--metrics_every','50',
 '--d_model','416','--n_layers','6','--n_heads','13','--n_kv_heads','1',
 '--gwtb_n_heads','4','--seq_len','512','--batch','12','--grad_accum','4',
 '--lr','6e-4','--warmup_steps','200','--steps','20000',
 '--log_every','100','--eval_every','500','--eval_batches','40','--save_every','1000',
 '--competitive_gwtb','--n_bids','3',
 '--world_model','--world_model_weight','0.01','--world_model_grad_clip','1.0']+resume
subprocess.run(cmd, check=True)

# 7) 生成 serve.pt
src=f'{CKPT}/final.pt' if os.path.exists(f'{CKPT}/final.pt') else f'{CKPT}/last.pt'
ck=torch.load(src, map_location='cpu', weights_only=False)
slim={'config':ck['config'],'model_state':ck['model_state'],'step':ck.get('step'),'loss':ck.get('loss')}
torch.save(slim, f'{CKPT}/serve.pt')
print("OK -> serve.pt  step=",slim['step']," loss=",slim['loss'])
from google.colab import files; files.download(f'{CKPT}/serve.pt')

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CalledProcessError: Command '['git', 'clone', '--depth', '1', 'https://github.com/everest-an/M1.git', '/content/M1']' returned non-zero exit status 128.